<a href="https://colab.research.google.com/github/abhinayakscs25-svg/BIS/blob/main/bis_lab%203%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import random
import math

# -------------------------------
# 1. Problem data
# -------------------------------

# Node 0 is the depot; nodes 1 to 6 are customers.
distance = [
    [0, 4, 6, 8, 7, 5, 9],
    [4, 0, 3, 7, 6, 4, 8],
    [6, 3, 0, 4, 5, 7, 6],
    [8, 7, 4, 0, 3, 6, 5],
    [7, 6, 5, 3, 0, 4, 3],
    [5, 4, 7, 6, 4, 0, 5],
    [9, 8, 6, 5, 3, 5, 0]
]

demand = [0, 2, 2, 3, 2, 1, 2]
capacity = 5

# Earliest and latest preferred service times
earliest = [0, 5, 8, 10, 7, 12, 14]
latest =   [0, 10, 13, 15, 12, 17, 19]

# Algorithm parameters
ANTS = 20
ITERATIONS = 100
ALPHA = 1.0
BETA = 2.0
EVAPORATION = 0.5
PHEROMONE_CONSTANT = 100
WAIT_COST = 1
LATE_PENALTY = 5
VEHICLE_COST = 10

N = len(distance)
random.seed(42)

# -------------------------------
# 2. Initialize pheromones
# -------------------------------

pheromone = [
    [1.0 if i != j else 0.0 for j in range(N)]
    for i in range(N)
]


# -------------------------------
# 3. Construct routes using ants
# -------------------------------

def construct_routes():
    unvisited = set(range(1, N))
    routes = []

    while unvisited:
        route = [0]
        current = 0
        load = 0

        while True:
            candidates = [
                c for c in unvisited
                if load + demand[c] <= capacity
            ]

            if not candidates:
                break

            weights = []
            for c in candidates:
                tau = pheromone[current][c] ** ALPHA
                eta = (1 / distance[current][c]) ** BETA
                weights.append(tau * eta)

            nxt = random.choices(
                candidates, weights=weights, k=1
            )[0]

            route.append(nxt)
            unvisited.remove(nxt)
            load += demand[nxt]
            current = nxt

        route.append(0)
        routes.append(route)

    return routes


# -------------------------------
# 4. Calculate total route cost
# -------------------------------

def calculate_cost(routes):
    total = 0

    for route in routes:
        total += VEHICLE_COST
        time = 0

        for i in range(len(route) - 1):
            a, b = route[i], route[i + 1]
            travel = distance[a][b]

            total += travel
            time += travel

            if b != 0:
                if time < earliest[b]:
                    total += WAIT_COST * (
                        earliest[b] - time
                    )
                    time = earliest[b]

                elif time > latest[b]:
                    total += LATE_PENALTY * (
                        time - latest[b]
                    )

    return total


# -------------------------------
# 5. Brainstorming-inspired update
# -------------------------------

def brainstorm(routes):
    # Create a new solution by swapping two customers
    new_routes = [r[:] for r in routes]

    customer_positions = [
        (i, j)
        for i, r in enumerate(new_routes)
        for j in range(1, len(r) - 1)
    ]

    if len(customer_positions) >= 2:
        (r1, p1), (r2, p2) = random.sample(
            customer_positions, 2
        )

        new_routes[r1][p1], new_routes[r2][p2] = (
            new_routes[r2][p2],
            new_routes[r1][p1]
        )

    # Keep the change only if capacity remains valid
    if all(
        sum(demand[c] for c in r[1:-1]) <= capacity
        for r in new_routes
    ):
        if calculate_cost(new_routes) < calculate_cost(routes):
            return new_routes

    return routes


# -------------------------------
# 6. Main IBSO-ACO loop
# -------------------------------

best_routes = None
best_cost = math.inf

for iteration in range(ITERATIONS):
    solutions = []

    for ant in range(ANTS):
        routes = construct_routes()
        routes = brainstorm(routes)
        cost = calculate_cost(routes)
        solutions.append((routes, cost))

        if cost < best_cost:
            best_routes = routes
            best_cost = cost

    # Evaporate pheromones
    for i in range(N):
        for j in range(N):
            pheromone[i][j] *= (1 - EVAPORATION)

    # Deposit more pheromone for lower-cost solutions
    for routes, cost in solutions:
        deposit = PHEROMONE_CONSTANT / max(cost, 1)

        for route in routes:
            for i in range(len(route) - 1):
                a, b = route[i], route[i + 1]
                pheromone[a][b] += deposit
                pheromone[b][a] += deposit

    print(
        f"Iteration {iteration + 1}: "
        f"Best cost = {best_cost:.2f}"
    )

# -------------------------------
# 7. Display the final solution
# -------------------------------

print("\nBest vehicle routes:")
for i, route in enumerate(best_routes, start=1):
    print(f"Vehicle {i}: {' -> '.join(map(str, route))}")

print("Minimum total cost:", round(best_cost, 2))
print("Vehicles required:", len(best_routes))


Iteration 1: Best cost = 88.00
Iteration 2: Best cost = 88.00
Iteration 3: Best cost = 88.00
Iteration 4: Best cost = 88.00
Iteration 5: Best cost = 88.00
Iteration 6: Best cost = 88.00
Iteration 7: Best cost = 88.00
Iteration 8: Best cost = 88.00
Iteration 9: Best cost = 88.00
Iteration 10: Best cost = 88.00
Iteration 11: Best cost = 88.00
Iteration 12: Best cost = 88.00
Iteration 13: Best cost = 88.00
Iteration 14: Best cost = 88.00
Iteration 15: Best cost = 88.00
Iteration 16: Best cost = 88.00
Iteration 17: Best cost = 88.00
Iteration 18: Best cost = 85.00
Iteration 19: Best cost = 85.00
Iteration 20: Best cost = 85.00
Iteration 21: Best cost = 85.00
Iteration 22: Best cost = 85.00
Iteration 23: Best cost = 85.00
Iteration 24: Best cost = 85.00
Iteration 25: Best cost = 85.00
Iteration 26: Best cost = 85.00
Iteration 27: Best cost = 85.00
Iteration 28: Best cost = 85.00
Iteration 29: Best cost = 85.00
Iteration 30: Best cost = 85.00
Iteration 31: Best cost = 85.00
Iteration 32: Bes

In [4]:

import numpy as np

# Sample dataset containing three groups of points
X = np.array([
    [1.0, 1.0], [1.2, 0.8], [0.8, 1.2],
    [5.0, 5.0], [5.2, 4.8], [4.8, 5.2],
    [9.0, 1.0], [9.2, 0.8], [8.8, 1.2]
])

C = 3                   # Number of clusters
m = 2.0                 # Fuzziness parameter
particles = 15          # Number of PSO particles
iterations = 100        # Maximum iterations

# Step 1: Calculate density of each data point
def calculate_density(X):
    dist_sq = np.sum(
        (X[:, None, :] - X[None, :, :]) ** 2, axis=2
    )
    fd = 1.0 / (np.mean(dist_sq) + 1e-10)
    density = np.sum(1.0 / (1.0 + fd * dist_sq), axis=1)
    return density, fd


# Step 2: Select initial centers using density
def select_centers(X, C):
    density, fd = calculate_density(X)
    scores = density.copy()
    centers = []

    for _ in range(C):
        idx = np.argmax(scores)
        center = X[idx]
        centers.append(center)

        dist_sq = np.sum((X - center) ** 2, axis=1)
        scores -= density[idx] / (1.0 + fd * dist_sq)
        scores[idx] = -np.inf

    return np.array(centers)


# Step 3: Calculate fuzzy membership values
def membership(X, centers):
    dist = np.linalg.norm(
        X[:, None, :] - centers[None, :, :], axis=2
    )
    dist = np.maximum(dist, 1e-10)

    power = 2.0 / (m - 1.0)
    weights = dist ** (-power)
    return weights / np.sum(weights, axis=1, keepdims=True)


# Step 4: Calculate clustering objective function
def fitness(X, centers):
    U = membership(X, centers)
    dist_sq = np.sum(
        (X[:, None, :] - centers[None, :, :]) ** 2, axis=2
    )
    return np.sum((U ** m) * dist_sq)


# Step 5: Initialize PSO particles
initial_centers = select_centers(X, C)
rng = np.random.default_rng(42)

positions = np.array([
    initial_centers + rng.normal(0, 0.5, initial_centers.shape)
    for _ in range(particles)
])
velocities = np.zeros_like(positions)

personal_best = positions.copy()
personal_scores = np.array([
    fitness(X, p) for p in positions
])

best_idx = np.argmin(personal_scores)
global_best = personal_best[best_idx].copy()
global_score = personal_scores[best_idx]

# Step 6: Optimize the cluster centers using PSO
for t in range(iterations):
    w = 0.9 - 0.5 * t / iterations

    for i in range(particles):
        r1 = rng.random(positions[i].shape)
        r2 = rng.random(positions[i].shape)

        velocities[i] = (
            w * velocities[i]
            + 1.5 * r1 * (personal_best[i] - positions[i])
            + 1.5 * r2 * (global_best - positions[i])
        )

        positions[i] += velocities[i]
        score = fitness(X, positions[i])

        if score < personal_scores[i]:
            personal_scores[i] = score
            personal_best[i] = positions[i].copy()

        if score < global_score:
            global_score = score
            global_best = positions[i].copy()

# Step 7: Display the final clustering results
U = membership(X, global_best)
labels = np.argmax(U, axis=1)

print("Final cluster centers:")
print(np.round(global_best, 2))

print("\nCluster assignment for each data point:")
for i in range(len(X)):
    print(
        f"Point {X[i]} -> Cluster {labels[i] + 1}, "
        f"Memberships: {np.round(U[i], 2)}"
    )

print("\nFinal objective function value:", round(global_score, 4))

Final cluster centers:
[[5. 5.]
 [1. 1.]
 [9. 1.]]

Cluster assignment for each data point:
Point [1. 1.] -> Cluster 2, Memberships: [0. 1. 0.]
Point [1.2 0.8] -> Cluster 2, Memberships: [0. 1. 0.]
Point [0.8 1.2] -> Cluster 2, Memberships: [0. 1. 0.]
Point [5. 5.] -> Cluster 1, Memberships: [1. 0. 0.]
Point [5.2 4.8] -> Cluster 1, Memberships: [0.99 0.   0.  ]
Point [4.8 5.2] -> Cluster 1, Memberships: [1. 0. 0.]
Point [9. 1.] -> Cluster 3, Memberships: [0. 0. 1.]
Point [9.2 0.8] -> Cluster 3, Memberships: [0. 0. 1.]
Point [8.8 1.2] -> Cluster 3, Memberships: [0. 0. 1.]

Final objective function value: 0.478
